In [1]:
from pathlib import Path
import datetime
import configparser

import pandas as pd
import xarray as xr
import s3fs

# from prefect import flow, get_run_logger

from utils import round_up_mins, get_slice_start_end_times


In [33]:
flow_start_time = "2025-06-17T02:40:00+00:00"

In [34]:
curr_time_offset = (
    datetime.datetime.now(datetime.timezone.utc)
    - (
        datetime.datetime.fromisoformat(flow_start_time)
        .astimezone(datetime.timezone.utc)
    )
)

In [35]:
time_offset_seconds = curr_time_offset.total_seconds()
slice_mins = 180
cred_file = "/home/exouser/.config/rclone/rclone.conf"
path_MVBS = "agr230002-bucket01/prefect_sh2506_test/acoustics/MVBS"
path_cache = "/media/volume/shimada_202506_volume/viz_data_cache"
file_MVBS_csv = "MVBS_files.csv"
file_MVBS_zarr = "latest_MVBS.zarr"

In [36]:
end_time = round_up_mins(
    datetime.datetime.now() - datetime.timedelta(seconds=time_offset_seconds),
    slice_mins=slice_mins,
).astimezone(datetime.timezone.utc)  # convert to UTC

print(
    "flow started with parameters:\n"
    f"- end_time: {end_time}\n"
    f"- slice_mins: {slice_mins}\n"
)


flow started with parameters:
- end_time: 2025-06-17 03:00:00+00:00
- slice_mins: 180



In [38]:
# Compute slice time range
start_time, end_time = get_slice_start_end_times(
    end_time=end_time, slice_mins=slice_mins, num_slices=1
)
start_time, end_time

([Timestamp('2025-06-17 00:00:00+0000', tz='UTC')],
 [Timestamp('2025-06-17 03:00:00+0000', tz='UTC')])

In [39]:
# Get cloud bucket
config = configparser.ConfigParser()
config.read(cred_file)
fs = s3fs.S3FileSystem(
    key=config["osn_sdsc_hake"]["access_key_id"],
    secret=config["osn_sdsc_hake"]["secret_access_key"],
    client_kwargs={"endpoint_url": config["osn_sdsc_hake"]["endpoint"]},
)

In [40]:
# Get all MVBS files in the bucket
MVBS_all = fs.glob(f"{path_MVBS}/*.zarr")
MVBS_all = sorted([Path(f).name for f in MVBS_all])
print(f"All MVBS files: {MVBS_all}")

All MVBS files: ['MVBS_20250613T115000.zarr', 'MVBS_20250613T120000.zarr', 'MVBS_20250613T121000.zarr', 'MVBS_20250613T122000.zarr', 'MVBS_20250613T123000.zarr', 'MVBS_20250613T124000.zarr', 'MVBS_20250613T125000.zarr', 'MVBS_20250613T130000.zarr', 'MVBS_20250613T131000.zarr', 'MVBS_20250613T132000.zarr', 'MVBS_20250613T133000.zarr', 'MVBS_20250613T134000.zarr', 'MVBS_20250613T135000.zarr', 'MVBS_20250613T140000.zarr', 'MVBS_20250613T141000.zarr', 'MVBS_20250613T142000.zarr', 'MVBS_20250613T143000.zarr', 'MVBS_20250613T144000.zarr', 'MVBS_20250613T145000.zarr', 'MVBS_20250613T150000.zarr', 'MVBS_20250613T151000.zarr', 'MVBS_20250613T152000.zarr', 'MVBS_20250613T153000.zarr', 'MVBS_20250613T154000.zarr', 'MVBS_20250613T155000.zarr', 'MVBS_20250613T160000.zarr', 'MVBS_20250613T161000.zarr', 'MVBS_20250613T162000.zarr', 'MVBS_20250613T163000.zarr', 'MVBS_20250613T164000.zarr', 'MVBS_20250613T165000.zarr', 'MVBS_20250613T170000.zarr', 'MVBS_20250613T171000.zarr', 'MVBS_20250613T172000.zarr

In [41]:
Path(path_MVBS).parent

PosixPath('agr230002-bucket01/prefect_sh2506_test/acoustics')

In [42]:
mapper = fs.get_mapper(str(Path(path_MVBS).parent / file_MVBS_csv))

In [43]:
# Load MVBS info dataframe
with fs.open(str(Path(path_MVBS).parent / file_MVBS_csv), "r") as f:
    df_MVBS = pd.read_csv(
        f,
        parse_dates=["first_ping_time", "last_ping_time"],
        index_col=0,
    )

In [44]:
df_MVBS

,MVBS_filename,first_ping_time,last_ping_time
0,MVBS_20250613T115000.zarr,2025-06-13 11:51:50,2025-06-13 11:59:55
1,MVBS_20250613T120000.zarr,2025-06-13 12:00:00,2025-06-13 12:09:55
2,MVBS_20250613T121000.zarr,2025-06-13 12:10:00,2025-06-13 12:19:55
3,MVBS_20250613T122000.zarr,2025-06-13 12:20:00,2025-06-13 12:29:55
4,MVBS_20250613T123000.zarr,2025-06-13 12:30:00,2025-06-13 12:39:55
...,...,...,...
237,MVBS_20250617T014000.zarr,2025-06-17 01:40:00,2025-06-17 01:59:55
238,MVBS_20250617T020000.zarr,2025-06-17 02:00:00,2025-06-17 02:19:55
239,MVBS_20250617T022000.zarr,2025-06-17 02:20:00,2025-06-17 02:39:55
240,MVBS_20250617T024000.zarr,2025-06-17 02:40:00,2025-06-17 02:59:55


In [45]:
# Convert last_ping_time and first_ping_time to UTC
if not df_MVBS.empty:
    if df_MVBS["last_ping_time"].dt.tz is None:
        df_MVBS["last_ping_time"] = df_MVBS["last_ping_time"].dt.tz_localize("UTC")
    if df_MVBS["first_ping_time"].dt.tz is None:
        df_MVBS["first_ping_time"] = df_MVBS["first_ping_time"].dt.tz_localize("UTC")

In [46]:
# Get MVBS files in the specified time range (only 1 slice)
MVBS_filenames = sorted(
    df_MVBS[
        (pd.to_datetime(df_MVBS["last_ping_time"]) >= start_time[0]) &
        (pd.to_datetime(df_MVBS["first_ping_time"]) <= end_time[0])
    ]["MVBS_filename"].tolist()
)

In [47]:
MVBS_filenames

['MVBS_20250617T000000.zarr',
 'MVBS_20250617T002000.zarr',
 'MVBS_20250617T004000.zarr',
 'MVBS_20250617T010000.zarr',
 'MVBS_20250617T012000.zarr',
 'MVBS_20250617T014000.zarr',
 'MVBS_20250617T020000.zarr',
 'MVBS_20250617T022000.zarr',
 'MVBS_20250617T024000.zarr',
 'MVBS_20250617T030000.zarr']

In [48]:
# Assmeble fs mapper for the MVBS files
MVBS_filenames = [
    fs.get_mapper(str(Path(path_MVBS) / mvbsf)) for mvbsf in MVBS_filenames
]

In [49]:
MVBS_filenames

In [50]:
# Combine and prepare MVBS dataset
ds_MVBS = xr.open_mfdataset(
    MVBS_filenames,
    parallel=True,
    coords="minimal",
    data_vars="minimal",
    compat='override',
    chunks={"channel": -1, "ping_time": -1, "depth": -1},  # load everything into 1 chunk
    engine="zarr",  # use zarr engine for reading
)

In [51]:
ds_MVBS["echo_range"] = ds_MVBS["depth"]
ds_MVBS = ds_MVBS.swap_dims({"depth": "echo_range"})

# Add actual_range to allow using holoviz
ds_MVBS["Sv"] = ds_MVBS["Sv"].assign_attrs(
    actual_range=(float(ds_MVBS["Sv"].min().compute()),
                float(ds_MVBS["Sv"].max().compute()))
)

In [57]:
for var in ds_MVBS.data_vars:
    print(ds_MVBS[var].encoding)

{'chunks': (5, 240, 660), 'preferred_chunks': {'channel': 5, 'ping_time': 240, 'depth': 660}, 'compressor': Blosc(cname='lz4', clevel=5, shuffle=SHUFFLE, blocksize=0), 'filters': None, '_FillValue': nan, 'dtype': dtype('float64')}
{'chunks': (5,), 'preferred_chunks': {'channel': 5}, 'compressor': Blosc(cname='lz4', clevel=5, shuffle=SHUFFLE, blocksize=0), 'filters': None, '_FillValue': nan, 'dtype': dtype('float64')}
{'chunks': (240,), 'preferred_chunks': {'ping_time': 240}, 'compressor': Blosc(cname='lz4', clevel=5, shuffle=SHUFFLE, blocksize=0), 'filters': None, '_FillValue': nan, 'dtype': dtype('float64')}
{'chunks': (240,), 'preferred_chunks': {'ping_time': 240}, 'compressor': Blosc(cname='lz4', clevel=5, shuffle=SHUFFLE, blocksize=0), 'filters': None, '_FillValue': nan, 'dtype': dtype('float64')}


In [58]:
for var in ds_MVBS.data_vars:
    if "chunks" in ds_MVBS[var].encoding:
        ds_MVBS[var].encoding.pop("chunks")
    if "preferred_chunks" in ds_MVBS[var].encoding:
        ds_MVBS[var].encoding.pop("preferred_chunks")

In [59]:
for var in ds_MVBS.data_vars:
    print(ds_MVBS[var].encoding)

{'compressor': Blosc(cname='lz4', clevel=5, shuffle=SHUFFLE, blocksize=0), 'filters': None, '_FillValue': nan, 'dtype': dtype('float64')}
{'compressor': Blosc(cname='lz4', clevel=5, shuffle=SHUFFLE, blocksize=0), 'filters': None, '_FillValue': nan, 'dtype': dtype('float64')}
{'compressor': Blosc(cname='lz4', clevel=5, shuffle=SHUFFLE, blocksize=0), 'filters': None, '_FillValue': nan, 'dtype': dtype('float64')}
{'compressor': Blosc(cname='lz4', clevel=5, shuffle=SHUFFLE, blocksize=0), 'filters': None, '_FillValue': nan, 'dtype': dtype('float64')}


In [62]:
ds_MVBS.chunk(
    {"channel": -1, "ping_time": -1, "echo_range": -1}
).to_zarr(
    Path(path_cache) / "latest_MVBS.zarr",  # cache is local
    mode="w",
    consolidated=True,
)